# Demo 3 - Network beacon & lateral-movement detection

**Workshop:** F16 Advanced analytics (notebooks / ML) · **Pool:** Medium · **Time:** ~35 min

Two analytics that are natural in Spark and awkward in KQL:
- **Part A - Beaconing:** regular, low-volume outbound connections over long windows (a C2
  signature). The tell is *regularity over time* - a statistical property, not a filter.
- **Part B - Lateral movement:** unusual internal-to-internal fan-out (e.g. SMB/RDP).

> Requires the `DeviceNetworkEvents` table (Microsoft Defender for Endpoint). If it isn't
> present, treat this as a walkthrough.

## 1. Configuration + connect

Set the workspace and the lookback window, then read `DeviceNetworkEvents`.

Two things worth noting in this cell:

- The time filter is applied **here**, once, so both Part A and Part B inherit it. Without
  it you would scan the full lake history, which for this table can be years.
- The time column is `TimeGenerated`. The Defender advanced-hunting name `Timestamp` does
  not exist in the Sentinel schema, and some Microsoft samples use it and will not resolve
  against a Sentinel workspace.

In [ ]:
WORKSPACE = "your-workspace-name"   # <-- replace with your workspace name
LOOKBACK_DAYS = 14                   # bound the demo - the lake holds years of history

from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd

data_provider = MicrosoftSentinelProvider(spark)

# DeviceNetworkEvents in the Sentinel schema uses TimeGenerated (the Defender advanced
# hunting name, Timestamp, does not exist here).
net = (data_provider.read_table("DeviceNetworkEvents", WORKSPACE)
       .filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS")))
print("DeviceNetworkEvents columns:", len(net.columns))

## Part A - Beacon detection

Bucket connections into hours per (device, remote IP), then measure how *regular* each pair
is. Beacons appear across **many hours**, at **low average volume**, with **very low
variance** (stddev).

In [ ]:
hourly = (net.withColumn("HourBucket", F.date_trunc("hour", F.col("TimeGenerated")))
             .groupBy("DeviceName", "RemoteIP", "HourBucket")
             .agg(F.count("*").alias("ConnectionCount")))

stats = (hourly.groupBy("DeviceName", "RemoteIP")
               .agg(F.count("*").alias("HoursSeen"),
                    F.avg("ConnectionCount").alias("AvgConnPerHour"),
                    F.stddev("ConnectionCount").alias("StdDevConnPerHour")))

beacon_candidates = (stats
    .filter((F.col("HoursSeen") > 10) &
            (F.col("AvgConnPerHour") < 5) &
            (F.col("StdDevConnPerHour") < 1.0))
    .orderBy(F.desc("HoursSeen")))

print("Beacon candidates:", beacon_candidates.count())
beacon_candidates.show(20, truncate=False)

### Plot one candidate - a beacon looks like a steady heartbeat

Take the top candidate and plot its hourly connection count over time.

This is the picture that makes the case. Human-driven traffic is lumpy - bursts during the
working day, nothing overnight. An automated beacon is a flat line of near-identical small
values, hour after hour, including at 4am on a Sunday.

That flatness is exactly what the standard deviation filter in the previous cell was
selecting for. The chart is the same finding, made obvious.

In [ ]:
rows = beacon_candidates.limit(1).collect()
if rows:
    dev, ip = rows[0]["DeviceName"], rows[0]["RemoteIP"]
    series = (hourly.filter((F.col("DeviceName") == dev) & (F.col("RemoteIP") == ip))
                    .orderBy("HourBucket").toPandas())
    series["HourBucket"] = pd.to_datetime(series["HourBucket"])
    plt.figure(figsize=(12, 5))
    plt.plot(series["HourBucket"], series["ConnectionCount"], marker="o")
    plt.title(f"Outbound connections - {dev} -> {ip}")
    plt.xlabel("Time (hourly)")
    plt.ylabel("Connection count")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("No beacon candidates found - widen the window or relax thresholds.")

## Part B - Lateral movement

Focus on **internal-to-internal** connections (RFC1918 both ends) and surface high-count
pairs, which can indicate an attacker pivoting between endpoints.

In [ ]:
internal_ip = r"^(10\.\d{1,3}\.\d{1,3}\.\d{1,3}|192\.168\.\d{1,3}\.\d{1,3}|172\.(1[6-9]|2[0-9]|3[0-1])\.\d{1,3}\.\d{1,3})$"

internal = net.filter(F.col("RemoteIP").rlike(internal_ip) & F.col("LocalIP").rlike(internal_ip))

lateral = (internal.groupBy("LocalIP", "RemoteIP", "InitiatingProcessAccountName")
                   .agg(F.count("*").alias("ConnectionCount"))
                   .filter(F.col("ConnectionCount") > 10)   # tune vs known-good automation
                   .orderBy(F.desc("ConnectionCount")))

print("Suspicious internal pairs:", lateral.count())
lateral.show(20, truncate=False)

## Optional - persist beacon candidates for KQL hunting

Uncomment to write the candidate list to a custom lake-tier table so the SOC can query it
in KQL without re-running the notebook.

This is the pattern worth internalising: do the expensive scan-and-aggregate in the lake
tier where storage is cheap, then persist only the compact result. The heavy read stays
cheap and the useful output becomes queryable.

In [ ]:
# Uncomment to write the compact result to the lake tier for the SOC to hunt.
# run_id = data_provider.save_as_table(
#     beacon_candidates, "NetworkBeacons_SPRK", write_options={"mode": "overwrite"})
# print("Wrote NetworkBeacons_SPRK, run id:", run_id)

## Recap

- Beaconing is about **regularity**, not volume - a model-shaped question, so it lives in a
  notebook.
- Calibrate thresholds against known-good automation (backup, monitoring) to cut noise.
- Same pattern as Demo 2: compute in the lake tier, promote only the small result set.